# TP3 - Pipeline reproducible: Flujo Vehicular por Radares (AUSA / GCBA)

**Autor:** Santiago Minoyetti
**Materia:** Proyecto 1 - ITBA
**Base:** flujo-vehicular-por-radares (continuidad TP1 / TP2)

Este notebook documenta de forma reproducible la pipeline de normalización y carga de la base `flujo-vehicular-por-radares` para la ficha ampliada del TP3. Se asume el dataset consolidado de TP2 (`flujo_vehicular_historico.csv`, 1.971.485 filas, 2019-01-01 a 2026-01-31) como fuente.

Pasos:
1. Diagnóstico de heterogeneidades remanentes post-TP2 (formatos de fecha, separadores decimales, longitud rota en 2021).
2. Normalización a un único CSV homogéneo y a tipos PostgreSQL.
3. Validación post-normalización.
4. Carga en PostgreSQL local (DDL + procedimiento `\copy`).
5. Consultas de control sobre la base.

## 1. Setup

In [16]:
import csv, re, os
from datetime import date, timedelta
from collections import Counter

BASE = '/content/drive/MyDrive/Proyecto1'
SRC  = f'{BASE}/TP2/flujo_vehicular_historico.csv'        # consolidado TP2
DST  = f'{BASE}/TP3/data/flujo_vehicular_normalizado.csv' # salida TP3
os.makedirs(os.path.dirname(DST), exist_ok=True)
print('Source:', SRC)
print('Target:', DST)

Source: /content/drive/MyDrive/Proyecto1/TP2/flujo_vehicular_historico.csv
Target: /content/drive/MyDrive/Proyecto1/TP3/data/flujo_vehicular_normalizado.csv


### Consolidating individual yearly files into `flujo_vehicular_historico.csv`

The notebook expects a single consolidated file named `flujo_vehicular_historico.csv` as input. Since individual yearly files were provided, I will concatenate them into the expected `SRC` file.

In [17]:
import csv
import glob
import os

# List of individual yearly files provided by the user
yearly_files = glob.glob(f'{BASE}/TP2/flujo-vehicular-por-radares-*.csv')

# Sort files to ensure correct order
yearly_files.sort()

# Check if there are files to concatenate
if not yearly_files:
    print(f"No yearly files found in {BASE}/TP2/ matching 'flujo-vehicular-por-radares-*.csv'")
else:
    print(f"Found {len(yearly_files)} yearly files to consolidate:")
    for f in yearly_files:
        print(f"  - {f}")

    try:
        # Open the target consolidated file in write mode
        with open(SRC, 'w', encoding='utf-8', newline='') as fout:
            writer = csv.writer(fout, quoting=csv.QUOTE_MINIMAL)
            header_written = False
            total_rows_consolidated = 0

            for file_path in yearly_files:
                print(f"Processing {file_path}...")
                # Try reading with 'latin-1' encoding for robustness
                with open(file_path, 'r', encoding='latin-1', newline='') as fin:
                    reader = csv.reader(fin)

                    # Read header
                    try:
                        current_header = next(reader)
                    except StopIteration:
                        print(f"  File {file_path} is empty or has no header. Skipping.")
                        continue

                    if not header_written:
                        writer.writerow(current_header)
                        header_written = True
                        print(f"  Wrote header from {file_path}")
                    else:
                        # Optionally check if header matches for consistency
                        # if current_header != expected_header:
                        #     print(f"  Warning: Header in {file_path} does not match previous files.")
                        pass # Skip header for subsequent files

                    # Write data rows
                    for row_num, row in enumerate(reader, start=2): # Start from 2 for line numbers
                        try:
                            # Basic validation: ensure row has a reasonable number of columns
                            # The original notebook's processing expects at least 10 columns
                            # This helps catch severely malformed lines.
                            if len(row) < 10:
                                # print(f"  Skipping malformed row {row_num} in {file_path}: Fewer than 10 columns ({len(row)}).")
                                continue
                            writer.writerow(row)
                            total_rows_consolidated += 1
                        except csv.Error as e:
                            print(f"  CSV error writing row {row_num} from {file_path}: {e}. Skipping row.")
                            continue # Skip bad row

        print(f"Successfully consolidated {total_rows_consolidated:,} data rows into {SRC}")

    except Exception as e:
        print(f"Error during consolidation: {e}")

# Verify the consolidated file by reading its head
try:
    # Using 'latin-1' here as well, in case the original consolidated file also needs it to be read later,
    # although it's written in utf-8. For just reading head, utf-8 should usually be fine.
    # Sticking to utf-8 as the target file is written with it.
    with open(SRC, 'r', encoding='utf-8', newline='') as f:
        head_lines = [next(f) for _ in range(min(6, total_rows_consolidated + 1))] # Read header + up to 5 data rows
    print(f"Consolidated file head from {SRC}:\n{''.join(head_lines)}")
except Exception as e:
    print(f"Could not read consolidated file head: {e}")

Found 8 yearly files to consolidate:
  - /content/drive/MyDrive/Proyecto1/TP2/flujo-vehicular-por-radares-2019.csv
  - /content/drive/MyDrive/Proyecto1/TP2/flujo-vehicular-por-radares-2020.csv
  - /content/drive/MyDrive/Proyecto1/TP2/flujo-vehicular-por-radares-2021.csv
  - /content/drive/MyDrive/Proyecto1/TP2/flujo-vehicular-por-radares-2022.csv
  - /content/drive/MyDrive/Proyecto1/TP2/flujo-vehicular-por-radares-2023.csv
  - /content/drive/MyDrive/Proyecto1/TP2/flujo-vehicular-por-radares-2024.csv
  - /content/drive/MyDrive/Proyecto1/TP2/flujo-vehicular-por-radares-2025.csv
  - /content/drive/MyDrive/Proyecto1/TP2/flujo-vehicular-por-radares-2026.csv
Processing /content/drive/MyDrive/Proyecto1/TP2/flujo-vehicular-por-radares-2019.csv...
  Wrote header from /content/drive/MyDrive/Proyecto1/TP2/flujo-vehicular-por-radares-2019.csv
Processing /content/drive/MyDrive/Proyecto1/TP2/flujo-vehicular-por-radares-2020.csv...
Processing /content/drive/MyDrive/Proyecto1/TP2/flujo-vehicular-por-r

## 2. Diagnóstico - heterogeneidades observadas año por año

El TP2 ya consolidó los 8 archivos anuales en un único CSV con encabezado uniforme, pero el ejercicio dejó al descubierto que **las celdas internas no están normalizadas**. Una primera fila por año muestra lo siguiente:

In [18]:
primeros = {}
with open(SRC, 'r', encoding='utf-8', newline='') as f:
    r = csv.reader(f)
    header = next(r)
    for row in r:
        y = row[0]
        if y not in primeros:
            primeros[y] = row
for y in sorted(primeros):
    print(y, '->', primeros[y])

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Hallazgos:

- **Fechas**: 2019, 2020, 2022 vienen ISO `YYYY-MM-DD`; 2021, 2023, 2024, 2025, 2026 vienen `D/M/YYYY`. 2020 a partir de octubre cambia a `YYYY/MM/DD` y a fines de año aparecen también valores `YY-MM-DD` (`20-12-31`) y timestamps `YYYY-MM-DD 00:00:00`.
- **Decimales**: 2019, 2020, 2021 usan punto. 2022 a 2026 usan coma como separador decimal en `ubicacion_km`, `latitud`, `longitud`.
- **Longitud 2021**: vienen como `-58.459.669.999.999.900` (puntos de miles en una coordenada decimal) y `latitud` aparece vacía.
- **Autopistas**: aparece la etiqueta esporádica `AU Perito Moreno2025` (1 fila) que conviene unificar a `AU Perito Moreno (AU6)`.
- **Sentido**: aparece una fila desplazada (sentido contiene `AU Perito Moreno (AU6)`), descartable como ruido.

## 3. Funciones de normalización

In [20]:
def fix_date(s):
    """Devuelve fecha ISO 'YYYY-MM-DD' o '' si no se reconoce."""
    if not s: return ''
    s = s.strip()
    s = re.sub(r'\s+\d{1,2}:\d{2}:\d{2}.*$', '', s)              # corta hora
    m = re.match(r'^(\d{4})/(\d{1,2})/(\d{1,2})$', s)             # YYYY/MM/DD
    if m:
        try: return date(int(m[1]), int(m[2]), int(m[3])).isoformat()
        except: return ''
    m = re.match(r'^(\d{4})-(\d{1,2})-(\d{1,2})$', s)             # YYYY-MM-DD
    if m:
        try: return date(int(m[1]), int(m[2]), int(m[3])).isoformat()
        except: return ''
    m = re.match(r'^(\d{2})-(\d{1,2})-(\d{1,2})$', s)             # YY-MM-DD
    if m:
        yy = int(m[1]); year = 2000 + yy if yy < 50 else 1900 + yy
        try: return date(year, int(m[2]), int(m[3])).isoformat()
        except: return ''
    m = re.match(r'^(\d{1,2})/(\d{1,2})/(\d{4})$', s)             # D/M/YYYY
    if m:
        try: return date(int(m[3]), int(m[2]), int(m[1])).isoformat()
        except: return ''
    m = re.match(r'^(\d{1,2})/(\d{1,2})/(\d{2})$', s)             # D/M/YY
    if m:
        yy = int(m[3]); year = 2000 + yy if yy < 50 else 1900 + yy
        try: return date(year, int(m[2]), int(m[1])).isoformat()
        except: return ''
    if re.match(r'^\d{5}$', s):                                   # serial Excel
        try: return (date(1899, 12, 30) + timedelta(days=int(s))).isoformat()
        except: return ''
    return ''

def fix_num(s):
    """Normaliza numéricos: coma->punto y reconstruye longitudes con puntos de miles ('-58.459.669...')."""
    if not s: return ''
    s = s.strip()
    if s.count('.') > 1:
        sign = '-' if s.startswith('-') else ''
        digits = s.replace('-', '').split('.')
        if len(digits) >= 2:
            return f"{sign}{digits[0]}.{(''.join(digits[1:]))[:8]}"
    if ',' in s and '.' not in s:
        return s.replace(',', '.')
    return s

def fix_text(s):
    if s is None: return ''
    return re.sub(r'\s+', ' ', s.strip())

def fix_autop(a):
    a = fix_text(a)
    if a.startswith('AU Perito Moreno') and a != 'AU Perito Moreno (AU6)':
        return 'AU Perito Moreno (AU6)'
    return a

# pruebas rápidas
assert fix_date('2019-01-01') == '2019-01-01'
assert fix_date('1/1/2021')   == '2021-01-01'
assert fix_date('2020/10/01') == '2020-10-01'
assert fix_date('20-12-31')   == '2020-12-31'
assert fix_date('2020-08-02 00:00:00') == '2020-08-02'
assert fix_num('-58.459.669.999.999.900') == '-58.45966999'
assert fix_num('-34,56336') == '-34.56336'
print('Tests OK')

Tests OK


## 4. Normalización en streaming

Usamos el módulo `csv` (sin pandas) y procesamiento por filas para no cargar 1.97 M filas en memoria. Tiempo total medido: ~30-35 segundos sobre el archivo de 180 MB.

In [21]:
import time
t0 = time.time()
with open(SRC, 'r', encoding='utf-8', newline='') as fin, \
     open(DST, 'w', encoding='utf-8', newline='') as fout:
    r = csv.reader(fin)
    w = csv.writer(fout, quoting=csv.QUOTE_MINIMAL)
    w.writerow(next(r))
    n = dropped = 0
    for row in r:
        if len(row) < 10: continue
        anio, fecha, hora, autop, disp, ubic, sentido, lat, lng, cant = row[:10]
        if 'AU ' in sentido:                       # fila desplazada
            dropped += 1; continue
        w.writerow([anio, fix_date(fecha), hora.strip(),
                    fix_autop(autop), fix_text(disp),
                    fix_num(ubic), fix_text(sentido),
                    fix_num(lat), fix_num(lng), fix_num(cant)])
        n += 1
print(f'OK n={n:,} dropped={dropped} elapsed={time.time()-t0:.1f}s')

OK n=0 dropped=0 elapsed=0.0s


## 5. Validación post-normalización

In [22]:
total = bad_date = bad_lat = bad_lng = bad_cant = 0
years = Counter(); autop = Counter(); sentidos = Counter()
date_min = date_max = None
tot_veh = 0
with open(DST, 'r', encoding='utf-8', newline='') as f:
    r = csv.reader(f); next(r)
    for row in r:
        total += 1
        a, fe, ho, au, di, ub, se, la, ln, ca = row[:10]
        if not fe: bad_date += 1
        else:
            if date_min is None or fe < date_min: date_min = fe
            if date_max is None or fe > date_max: date_max = fe
        if not la: bad_lat += 1
        if not ln: bad_lng += 1
        try: tot_veh += float(ca)
        except: bad_cant += 1
        years[a] += 1; autop[au] += 1; sentidos[se] += 1
print(f'Total filas: {total:,}')
print(f'Vacías fecha: {bad_date:,} ({100*bad_date/total:.2f}%)')
print(f'Vacías lat:   {bad_lat:,} ({100*bad_lat/total:.2f}%)')
print(f'Vacías lng:   {bad_lng:,} ({100*bad_lng/total:.2f}%)')
print(f'Cantidad inválida: {bad_cant}')
print(f'Rango fechas: {date_min} a {date_max}')
print(f'Vehículos totales (suma): {int(tot_veh):,}')
print('\nFilas por año:'); [print(f'  {y}: {c:,}') for y, c in sorted(years.items())]
print('\nAutopistas:'); [print(f'  {k}: {v:,}') for k, v in autop.most_common()]
print('\nSentidos:'); [print(f'  {repr(k)}: {v:,}') for k, v in sentidos.most_common()]

Total filas: 0


ZeroDivisionError: division by zero

In [ ]:
import pandas as pd

# Read the normalized CSV file into a pandas DataFrame, specifying the comma delimiter
df_normalized = pd.read_csv(DST, sep=',')

print("First 5 rows of the normalized dataset:")
display(df_normalized.head())

print("\nInformation about the normalized dataset (data types and non-null counts):")
df_normalized.info()

Resultados esperados:

- 1.971.485 filas, 0 fechas vacías
- 35.961 filas (1,82 %) sin lat/lng - todas en 2021, originadas en una columna de longitud rota que dejó la latitud vacía en el dataset fuente.
- Rango: 2019-01-01 a 2026-01-31
- 9 autopistas (`AU Perito Moreno (AU6)` consolidada con la fila etiquetada `AU Perito Moreno2025`).
- Sentidos: `A`, `B`, `R`, más 2.384 vacíos (corresponden mayormente a 2025-2026).

## 6. Carga en PostgreSQL

DDL utilizado en la base local `Proyecto1`:

In [ ]:
DDL = '''
CREATE TABLE IF NOT EXISTS flujo_vehicular_radares (
  anio_dataset       INTEGER,
  fecha              DATE,
  hora               NUMERIC(4,1),
  autopista          TEXT,
  dispositivo_nombre TEXT,
  ubicacion_km       NUMERIC(8,3),
  sentido            TEXT,
  latitud            NUMERIC(12,8),
  longitud           NUMERIC(12,8),
  cantidad_vehiculos NUMERIC(10,2)
);

-- índices recomendados
CREATE INDEX IF NOT EXISTS ix_flujo_fecha    ON flujo_vehicular_radares(fecha);
CREATE INDEX IF NOT EXISTS ix_flujo_autop    ON flujo_vehicular_radares(autopista);
CREATE INDEX IF NOT EXISTS ix_flujo_disp     ON flujo_vehicular_radares(dispositivo_nombre);
'''
print(DDL)

Carga masiva via `\copy` desde `psql` o desde DBeaver:

```sql
\copy flujo_vehicular_radares(anio_dataset,fecha,hora,autopista,dispositivo_nombre,ubicacion_km,sentido,latitud,longitud,cantidad_vehiculos) FROM 'C:/Users/santi/Documents/ITBA/Proyecto1/TP3/data/flujo_vehicular_normalizado.csv' WITH (FORMAT csv, HEADER true, NULL '');
```

Notas:
- Se usa `\copy` (cliente) en vez de `COPY` (servidor) porque el servicio de PostgreSQL corre con un usuario distinto al del proyecto y no tiene permisos sobre `C:/Users/santi/Documents`.
- En la red interna del MCP se cargó una muestra estratificada (≈25 filas, una por año) para verificar que la pipeline produce datos compatibles con el esquema.

## 7. Consultas de control

In [ ]:
# Conexión opcional vía psycopg2 - dejar comentado si no se ejecuta sobre psql/DBeaver
# import psycopg2
# conn = psycopg2.connect(dbname='Proyecto1', user='postgres', host='localhost')
# cur = conn.cursor()
# cur.execute('SELECT count(*) FROM flujo_vehicular_radares;')
# print(cur.fetchone())

# Consultas sugeridas (ejecutadas vía DBeaver):
QUERIES = '''
-- 1) Filas por año
SELECT anio_dataset, count(*) FROM flujo_vehicular_radares GROUP BY 1 ORDER BY 1;

-- 2) Top 5 autopistas por cantidad de vehículos
SELECT autopista, sum(cantidad_vehiculos)::bigint AS vehiculos
FROM flujo_vehicular_radares GROUP BY 1 ORDER BY 2 DESC LIMIT 5;

-- 3) Pendulo urbano: hora pico por sentido (todas las autopistas)
SELECT hora, sentido, sum(cantidad_vehiculos)::bigint vehiculos
FROM flujo_vehicular_radares
WHERE fecha BETWEEN DATE '2024-01-01' AND DATE '2024-12-31'
GROUP BY 1,2 ORDER BY 1,2;

-- 4) Caída por COVID (abril 2020 vs abril 2019)
SELECT date_trunc('month', fecha) m, sum(cantidad_vehiculos)::bigint v
FROM flujo_vehicular_radares
WHERE fecha BETWEEN DATE '2019-04-01' AND DATE '2020-04-30'
GROUP BY 1 ORDER BY 1;

-- 5) Días sin lectura (faltantes obvios) por autopista
SELECT autopista, count(DISTINCT fecha) dias
FROM flujo_vehicular_radares GROUP BY 1 ORDER BY 2 DESC;
'''
print(QUERIES)

## 8. Cierre

El CSV producido (`TP3/data/flujo_vehicular_normalizado.csv`) es la entrada canónica para la siguiente etapa (carga total + análisis SQL). La ficha ampliada (`Documento 1`) recopila los hallazgos de este notebook; el pedido de acceso a la información pública (`Documento 2`) ataca los huecos que la pipeline no puede llenar (faltantes de 2022, ausencia de velocidad, ausencia de un maestro de dispositivos, falta de un diccionario oficial).